Preprocesamiento

In [18]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("D57000_complete.csv", sep=";")

df["Titulo"]      = df["Titulo"].astype(str)
df["Descripcion"] = df["Descripcion"].astype(str)

# Texto base = título + descripción
df["text"] = df["Titulo"] + " . " + df["Descripcion"]

texts = df["text"].tolist()

#Usamos solo las noticias reales

REAL_LABEL = 1       
df = df[df["Label"] == REAL_LABEL].reset_index(drop=True)

# Tokenización
max_words = 10000
tk = Tokenizer(num_words=max_words, oov_token="<UNK>")
tk.fit_on_texts(texts)

sequences = tk.texts_to_sequences(texts)
vocab_size = min(max_words, len(tk.word_index) + 1)




pares

In [19]:
import numpy as np

input_seqs = []
next_words = []

max_len = 10

STEP = 3  

for seq in sequences:
    for i in range(1, len(seq), STEP):
        in_seq = seq[max(0, i - max_len):i]
        target = seq[i]
        input_seqs.append(in_seq)
        next_words.append(target)

MAX_EXAMPLES = 100000

if len(input_seqs) > MAX_EXAMPLES:
    input_seqs = input_seqs[:MAX_EXAMPLES]
    next_words = next_words[:MAX_EXAMPLES]


X = pad_sequences(input_seqs, maxlen=max_len, padding="pre")
y = np.array(next_words)

RNN

In [20]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Model

embedding_dim = 100

inputs = Input(shape=(max_len,))
x = Embedding(input_dim=vocab_size, output_dim=embedding_dim)(inputs)
x = SimpleRNN(64)(x)
outputs = Dense(vocab_size, activation="softmax")(x)

rnn_model = Model(inputs, outputs)
rnn_model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

rnn_model.summary()

history_rnn = rnn_model.fit(
    X, y,
    batch_size=256,
    epochs=10,
    validation_split=0.1
)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 10, 100)        │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 64)             │        10,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,660,560 (6.33 MB)

 Trainable params: 1,660,560 (6.33 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.0676 - loss: 6.6633 - val_accuracy: 0.0851 - val_loss: 6.2866
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1111 - loss: 6.0411 - val_accuracy: 0.1174 - val_loss: 5.9405
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.1345 - loss: 5.7028 - val_accuracy: 0.1372 - val_loss: 5.6946
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.1570 - loss: 5.4179 - val_accuracy: 0.1513 - val_loss: 5.5300
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.1731 - loss: 5.1926 - val_accuracy: 0.1577 - val_loss: 5.4287
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.1868 - loss: 5.0089 - val_accuracy: 0.1659 - val_loss: 5.3666
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.1981 - loss: 4.8516 - val_accuracy: 0.1709 - val_loss: 5.3191
Epoch 8/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.2085 - loss: 4.7099 - val_accu

lstm

In [21]:
from tensorflow.keras.layers import LSTM

inputs = Input(shape=(max_len,))
x = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)(inputs)
x = LSTM(64)(x)
outputs = Dense(vocab_size, activation="softmax")(x)

lstm_model = Model(inputs, outputs)
lstm_model.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

lstm_model.summary()

history_lstm = lstm_model.fit(
    X, y,
    batch_size=256,
    epochs=10,
    validation_split=0.1
)


d:\Programming-General\GitHub\Proyecto1NLP\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_6 (Embedding)         │ (None, 10, 100)        │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,692,240 (6.46 MB)

 Trainable params: 1,692,240 (6.46 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.0672 - loss: 6.7103 - val_accuracy: 0.0631 - val_loss: 6.3051
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.1087 - loss: 6.1145 - val_accuracy: 0.1118 - val_loss: 6.0689
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.1219 - loss: 5.8555 - val_accuracy: 0.1173 - val_loss: 5.8803
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.1345 - loss: 5.6537 - val_accuracy: 0.1285 - val_loss: 5.7548
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.1445 - loss: 5.5057 - val_accuracy: 0.1370 - val_loss: 5.6743
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.1582 - loss: 5.3885 - val_accuracy: 0.1470 - val_loss: 5.6136
Epoch 7/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.1666 - loss: 5.2870 - val_accuracy: 0.1497 - val_loss: 5.5727
Epoch 8/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.1743 - loss: 5.1995 - val_accu

seq2seq

In [22]:
# Textos fuente y destino
src_texts = df["Descripcion"].astype(str).tolist()
tgt_texts = df["Titulo"].astype(str).tolist()

# tokens de inicio / fin
tgt_in_texts  = ["<sos> " + t for t in tgt_texts]
tgt_out_texts = [t + " <eos>" for t in tgt_texts]

# tokenizadores
src_tk = Tokenizer(num_words=max_words, oov_token="<UNK>")
tgt_tk = Tokenizer(num_words=max_words, oov_token="<UNK>")

src_tk.fit_on_texts(src_texts)
tgt_tk.fit_on_texts(tgt_in_texts + tgt_out_texts)

max_len_src = 50
max_len_tgt = 15

src_seq     = pad_sequences(src_tk.texts_to_sequences(src_texts),     maxlen=max_len_src, padding="post")
tgt_in_seq  = pad_sequences(tgt_tk.texts_to_sequences(tgt_in_texts),  maxlen=max_len_tgt, padding="post")
tgt_out_seq = pad_sequences(tgt_tk.texts_to_sequences(tgt_out_texts), maxlen=max_len_tgt, padding="post")

num_decoder_tokens = len(tgt_tk.word_index) + 1




In [24]:
from tensorflow.keras.layers import LSTM

latent_dim = 128
embedding_dim = 100

# Encoder
encoder_inputs = Input(shape=(max_len_src,))
enc_emb = Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len_src)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_len_tgt,))
dec_emb_layer = Embedding(input_dim=num_decoder_tokens, output_dim=embedding_dim, input_length=max_len_tgt)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

seq2seq_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
seq2seq_model.compile(optimizer="adam",
                      loss="sparse_categorical_crossentropy",
                      metrics=["accuracy"])

seq2seq_model.summary()

seq2seq_model.fit(
    [src_seq, tgt_in_seq],
    np.expand_dims(tgt_out_seq, -1),
    batch_size=128,
    epochs=10,
    validation_split=0.1
)


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 50, 100)   │  1,000,000 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 15, 100)   │  3,096,800 │ input_layer_11[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 128),     │    117,248 │ embedding_9[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 15, 128), │    117,248 │ embedding_10[0][… │
│                     │ (None, 128),      │            │ lstm_4[0][1],     │
│                     │ (None, 128)]      │            │ lstm_4[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 15, 30968) │  3,994,872 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,326,168 (31.76 MB)

 Trainable params: 8,326,168 (31.76 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 86s 358ms/step - accuracy: 0.1010 - loss: 6.4581 - val_accuracy: 0.1172 - val_loss: 5.8461
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 353ms/step - accuracy: 0.1774 - loss: 5.5811 - val_accuracy: 0.2205 - val_loss: 5.2870
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 352ms/step - accuracy: 0.2238 - loss: 5.1518 - val_accuracy: 0.2322 - val_loss: 5.0186
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 352ms/step - accuracy: 0.2402 - loss: 4.9212 - val_accuracy: 0.2506 - val_loss: 4.8367
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 352ms/step - accuracy: 0.2552 - loss: 4.7476 - val_accuracy: 0.2596 - val_loss: 4.7063
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 353ms/step - accuracy: 0.2623 - loss: 4.6148 - val_accuracy: 0.2647 - val_loss: 4.6091
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 352ms/step - accuracy: 0.2696 - loss: 4.5117 - val_accuracy: 0.2697 - val_loss: 4.5356
Epoch 8/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 352ms/step - accuracy: 0.2765 - loss: 4